# Diffusion equation with time-dependent Neumann boundary conditions

$$
\mathbb{S}_u
\begin{cases}
\Omega = [0, L_x] \\
u_{\text{N}}(x=0, t)=\epsilon\cos(\omega t) \\
u_{\text{N}}(x=L_x)=0 \\
\mathsf{D}=\mathsf{I} \\
R=-1 \\
J=1 \\
\end{cases}
$$

In [ ]:
import numpy as np
from lucifex.mesh import interval_mesh
from lucifex.fdm import (
    BE, finite_difference_order,
    FiniteDifference, FunctionSeries, ConstantSeries,
)
from lucifex.sim import run, Simulation
from lucifex.fem import Constant
from lucifex.solver import ibvp, evaluation, BoundaryConditions
from lucifex.plt import plot_line, create_animation, save_figure, display_animation
from lucifex.pde.diffusion import diffusion

def cosine_wave(t, eps, omega):
    return eps * np.sin(omega * float(t))

def create_simulation(
    Lx: float,
    Nx: int,
    dt: float,
    eps: float,
    omega: float,
    D_diff: FiniteDifference,
    D_reac: FiniteDifference,
) -> Simulation:
    order = finite_difference_order(D_diff.order, D_reac.order)
    store = 1
    mesh = interval_mesh(Lx, Nx)
    t = ConstantSeries(mesh, name='t', ics=0.0)
    dt = Constant(mesh, dt, name='dt')  
    u = FunctionSeries((mesh, 'P', 1), 'u', order, store, ics=0.0)
    uN = ConstantSeries(mesh, order=order, store=store, ics=cosine_wave(0.0, eps, omega), name = 'uD')
    uN_solver = evaluation(uN, cosine_wave, future=True)(t[0] + dt, eps, omega)
    bcs = BoundaryConditions(
        ('neumann', lambda x: x[0], uN[1]),
        ('neumann', lambda x: x[0] - Lx, 0.0),
    )
    d = Constant(mesh, 1.0, 'd')
    u_solver = ibvp(diffusion, bcs=bcs)(u, dt, d, D_diff)
    return Simulation([uN_solver, u_solver], t, dt)


Lx = 1.0
Nx = 100
dt = 0.01
eps = 0.1
omega = 20
simulation = create_simulation(Lx, Nx, dt, eps, omega, BE, BE)
n_stop = 50
run(simulation, n_stop)

u = simulation['u']

In [ ]:
u_min = np.min([np.min(dofs) for dofs in u.dofs_series])
u_max = np.max([np.max(dofs) for dofs in u.dofs_series])

title_series = [f'$t={t:.2f}$' for t in u.time_series]
anim = create_animation(
    plot_line,
    y_lims=(u_min, u_max),
    x_label='$x$', 
    y_label='$u(x,t)$'
)(u.series, title=title_series)
anim_path = save_figure('u(x,t)', return_path=True)(anim)

display_animation(anim_path)

In [ ]:
slc = slice(0, None, 2)
legend_labels=(min(u.time_series[slc]), max(u.time_series[slc]))
fig, ax = plot_line(u.series[slc], legend_labels, '$t$', cyc='jet', x_label='$x$', y_label='$u$')
save_figure('u(x,t)', thumbnail=True)(fig)